# Week 5 Notebook — 传统 AI Baseline 与安全筛查指标

**课程项目：** Physics-informed AI for three-phase unbalanced microgrid N-1 security screening  
**本周主题：** 从三相 N-1 标签出发，建立传统机器学习 baseline，并用安全导向指标评价筛查器。

本 Notebook 是教材型材料：它不只是训练模型，还会用 proof-style checks 验证数据表、特征选择、指标计算、阈值选择和交叉验证协议的正确性。

## 学习目标

完成后学生应能够：

1. 读取 Week 4 的 N-1 dataset，并合并 Week 3 的 base-case features；
2. 证明模型输入中没有 post-contingency target leakage；
3. 训练 AllUnsafe、EngineeringRule、LogisticRegression、RandomForest、GradientBoosting 和 MLP baselines；
4. 用 validation set 选择 high-recall screening threshold；
5. 计算 recall、false negative rate、missed violations、calls saved、AUPRC 和 top-k recall；
6. 完成 random holdout、scenario-group CV 和 contingency-group CV；
7. 导出 Week 5 论文图表和结果表。

## 教学主线

Week 4 已经生成如下样本：

$$
(x_t, c) \rightarrow y_{t,c},
$$

其中 \(x_t\) 是 base-case operating scenario，\(c\) 是 contingency。Week 5 的部署问题是：

> 只使用 post-contingency 三相潮流求解之前可获得的信息，能否用便宜的模型判断哪些 N-1 case 仍然需要送入精确 \texttt{runpp\_3ph()} 检查？

正类是 **unsafe**。因此，本周重点不是 generic accuracy，而是 **high recall + low false-negative rate + useful power-flow call reduction**。

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import random
import warnings
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = Path("/mnt/data/week5_outputs_complete") if Path("/mnt/data").exists() else PROJECT_ROOT / "week5_outputs_complete"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")


## 1. 读取 Week 4 标签与 Week 3 base-case features

优先使用：

```text
week4_outputs_complete/week4_ai_ready_dataset.csv
week3_outputs_complete/week3_basecase_ai_features.csv
```

Week 4 表中的 `min_vm_pu`、`max_vuf_percent` 等是 **N-1 后结果**，不能作为 Week 5 输入。Week 3 的 base-case features 会被加上 `base_` 前缀后合并，因为它们是在 contingency 之前可获得的 N-0 信息。

In [ ]:
def first_existing_path(candidates: List[Path]) -> Optional[Path]:
    for p in candidates:
        if p.exists():
            return p
    return None

week4_ai_path = first_existing_path([
    PROJECT_ROOT / "week4_outputs_complete" / "week4_ai_ready_dataset.csv",
    PROJECT_ROOT / ".." / "week4_outputs_complete" / "week4_ai_ready_dataset.csv",
    Path("/mnt/data/week4_outputs_complete/week4_ai_ready_dataset.csv"),
])
week3_features_path = first_existing_path([
    PROJECT_ROOT / "week3_outputs_complete" / "week3_basecase_ai_features.csv",
    PROJECT_ROOT / ".." / "week3_outputs_complete" / "week3_basecase_ai_features.csv",
    Path("/mnt/data/week3_outputs_complete/week3_basecase_ai_features.csv"),
])

print("Week 4 dataset:", week4_ai_path)
print("Week 3 base-case features:", week3_features_path)
assert week4_ai_path is not None, "Week 4 AI-ready dataset is required for Week 5."
assert week3_features_path is not None, "Week 3 base-case feature table is required for Week 5."

n1 = pd.read_csv(week4_ai_path)
base = pd.read_csv(week3_features_path)
print(f"Week 4 rows: {len(n1)}; Week 3 scenarios: {len(base)}")
print("Week 4 columns:", len(n1.columns))
print("Week 3 columns:", len(base.columns))


In [ ]:
# Prefix base-case columns before merging to avoid confusing them with post-contingency metrics.
base_prefix = base.copy()
base_prefix = base_prefix.rename(columns={c: f"base_{c}" for c in base_prefix.columns if c != "scenario_id"})

df = n1.merge(base_prefix, on="scenario_id", how="left", validate="many_to_one")
df.insert(0, "row_id", [f"r{i:04d}" for i in range(len(df))])

# Make the label explicitly integer-valued for sklearn.
df["violation_label"] = df["violation_label"].astype(bool).astype(int)
df["severity_score"] = df["severity_score"].astype(float)

# Export the raw merged table for transparency.
df.to_csv(OUTPUT_DIR / "week5_ml_input_dataset.csv", index=False)
print("Merged ML input shape:", df.shape)
print(df[["row_id", "scenario_id", "contingency_id", "contingency_type", "violation_label", "severity_score"]].head().to_string(index=False))


## 2. Dataset schema proof

训练模型之前先检查数据表：

1. target column 存在；
2. target 同时包含 safe 和 unsafe 两类；
3. scenario 与 contingency 字段存在；
4. 每个 \(scenario, contingency\) pair 唯一；
5. severity 非负；
6. unsafe label 与 severity / violation flags 逻辑一致。

In [ ]:
TARGET_COL = "violation_label"
SEVERITY_COL = "severity_score"

required_cols = [
    "row_id", "scenario_id", "contingency_id", "contingency_type",
    "element_table", "element_index", "load_scale", "phase_a_factor",
    "phase_b_factor", "phase_c_factor", "pv_scale", "bess_p_discharge_mw",
    TARGET_COL, SEVERITY_COL,
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"
assert df["row_id"].is_unique, "row_id must be unique."
assert not df.duplicated(["scenario_id", "contingency_id"]).any(), "Duplicated scenario-contingency pair found."
assert set(df[TARGET_COL].dropna().unique()).issubset({0, 1}), "Target must be binary."
assert df[TARGET_COL].nunique() == 2, "Dataset must contain both safe and unsafe samples."
assert (df[SEVERITY_COL] >= 0).all(), "Severity must be nonnegative."

violation_flag_cols = [c for c in df.columns if c.endswith("_violation")]
unsafe = df[TARGET_COL].astype(bool)
flag_or_severity = (df[violation_flag_cols].fillna(False).astype(bool).any(axis=1)) | (df[SEVERITY_COL] > 0)
assert (unsafe <= flag_or_severity).all(), "Unsafe rows should have a violation flag or positive severity."

label_distribution = df.groupby(["contingency_type", TARGET_COL]).size().unstack(fill_value=0).reset_index()
label_distribution.to_csv(OUTPUT_DIR / "week5_label_distribution_by_type.csv", index=False)
print("Dataset schema proof passed.")
print("Label counts:", df[TARGET_COL].value_counts().to_dict())
print(label_distribution.to_string(index=False))


## 3. 构造 AI features 并防止 target leakage

最重要的 Week 5 proof 是 leakage guard。

允许作为输入：

- scenario settings；
- Week 3 base-case metrics；
- contingency metadata。

禁止作为输入：

- N-1 后 voltage / loading / VUF；
- convergence、service-loss、violation flags；
- final label 与 severity score。

In [ ]:
scenario_feature_cols = [
    "load_scale", "phase_a_factor", "phase_b_factor", "phase_c_factor",
    "pv_scale", "bess_p_discharge_mw",
]
contingency_feature_cols = ["contingency_type", "element_table", "element_index"]
basecase_feature_cols = [
    c for c in df.columns
    if c.startswith("base_") and c not in {"base_converged", "base_mode"}
]
feature_cols = scenario_feature_cols + contingency_feature_cols + basecase_feature_cols + ["base_mode"]
feature_cols = [c for c in feature_cols if c in df.columns]

forbidden_tokens = [
    "converged", "min_vm", "max_vm", "max_vuf", "max_line_loading",
    "unserved", "critical_unserved", "violation", "severity", "pf_status",
]
# Base-case metrics are allowed even if their names include voltage/loading/VUF tokens.
leaky_features = [
    c for c in feature_cols
    if (not c.startswith("base_")) and any(tok in c for tok in forbidden_tokens)
]
assert not leaky_features, f"Feature leakage detected: {leaky_features}"

X = df[feature_cols].copy()
y = df[TARGET_COL].astype(int).copy()

categorical_cols = [c for c in X.columns if X[c].dtype == "object" or c in ["contingency_type", "element_table", "base_mode"]]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

feature_table = pd.DataFrame({
    "feature": feature_cols,
    "type": ["categorical" if c in categorical_cols else "numeric" for c in feature_cols],
})
feature_table.to_csv(OUTPUT_DIR / "week5_feature_list.csv", index=False)
df[["row_id", "scenario_id", "contingency_id", TARGET_COL, SEVERITY_COL] + feature_cols].to_csv(
    OUTPUT_DIR / "week5_baseline_dataset.csv", index=False
)

print("No-leakage feature proof passed.")
print(f"Number of raw features before one-hot encoding: {len(feature_cols)}")
print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


## 4. 安全导向指标

正类是 **unsafe**。核心指标包括：

$$
\text{Recall}=\frac{TP}{TP+FN},
\quad
\text{FNR}=\frac{FN}{TP+FN},
\quad
\text{CallsSaved}=\frac{TN+FN}{N}.
$$

如果模型节省了很多潮流计算，但 FN 很多，就不适合安全筛查。

In [ ]:
def safe_roc_auc(y_true: Iterable[int], proba: Iterable[float]) -> float:
    y_arr = np.asarray(y_true).astype(int)
    if len(np.unique(y_arr)) < 2:
        return float("nan")
    return float(roc_auc_score(y_arr, proba))


def safe_auprc(y_true: Iterable[int], proba: Iterable[float]) -> float:
    y_arr = np.asarray(y_true).astype(int)
    if len(np.unique(y_arr)) < 2:
        return float("nan")
    return float(average_precision_score(y_arr, proba))


def compute_metrics(y_true: Iterable[int], proba: Iterable[float], threshold: float) -> Dict[str, float]:
    y_arr = np.asarray(y_true).astype(int)
    p_arr = np.asarray(proba).astype(float)
    pred = (p_arr >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_arr, pred, labels=[0, 1]).ravel()
    n = len(y_arr)
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else float("nan")
    return {
        "threshold": float(threshold),
        "n": int(n),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "accuracy": float((tp + tn) / n),
        "precision": float(precision),
        "recall": float(recall),
        "fnr": float(1 - recall) if not math.isnan(recall) else float("nan"),
        "specificity": float(specificity),
        "roc_auc": safe_roc_auc(y_arr, p_arr),
        "auprc": safe_auprc(y_arr, p_arr),
        "brier": float(brier_score_loss(y_arr, np.clip(p_arr, 0, 1))),
        "calls_saved_fraction": float((tn + fn) / n),
        "workload_fraction": float((tp + fp) / n),
        "missed_violations": int(fn),
        "predicted_unsafe": int(tp + fp),
        "predicted_safe": int(tn + fn),
    }


def choose_high_recall_threshold(y_true: Iterable[int], proba: Iterable[float], target_recall: float = 0.95) -> Tuple[float, pd.DataFrame]:
    y_arr = np.asarray(y_true).astype(int)
    p_arr = np.asarray(proba).astype(float)
    thresholds = np.unique(np.concatenate([[0.0], p_arr, [1.0]]))
    rows = []
    for tau in thresholds:
        rows.append(compute_metrics(y_arr, p_arr, threshold=float(tau)))
    table = pd.DataFrame(rows)
    feasible = table[table["recall"] >= target_recall].copy()
    if len(feasible) == 0:
        # Fall back to the threshold with highest recall, then lowest false negatives.
        chosen = table.sort_values(["recall", "calls_saved_fraction"], ascending=[False, False]).iloc[0]
    else:
        # Among high-recall thresholds, save the most power-flow calls. If tied, prefer higher precision.
        chosen = feasible.sort_values(["calls_saved_fraction", "precision"], ascending=[False, False]).iloc[0]
    return float(chosen["threshold"]), table

print("Metric helper proof passed on a toy vector:")
toy_y = np.array([0, 0, 1, 1])
toy_p = np.array([0.1, 0.8, 0.7, 0.9])
print(compute_metrics(toy_y, toy_p, threshold=0.5))


## 5. 定义 baseline models

本周包含两个非学习 baseline 和四个学习 baseline：

1. **AllUnsafe**：所有 contingency 都送入三相潮流，recall=1，但 calls saved=0；
2. **EngineeringRule**：基于 outage type、base voltage、base loading 和 VUF 的工程规则；
3. **LogisticRegression**：可解释线性模型；
4. **RandomForest**：非线性 tabular baseline；
5. **GradientBoosting**：强 tabular baseline；
6. **MLP**：为 Week 6 的 PI-MLP 做铺垫。

In [ ]:
class AllUnsafeClassifier(BaseEstimator, ClassifierMixin):
    def fit(self, X, y=None):
        return self
    def predict_proba(self, X):
        n = len(X)
        return np.column_stack([np.zeros(n), np.ones(n)])


class EngineeringRuleClassifier(BaseEstimator, ClassifierMixin):
    """Deterministic engineering rule baseline."""
    def fit(self, X, y=None):
        return self

    def predict_proba(self, X):
        Xd = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        n = len(Xd)
        risk = np.full(n, 0.10, dtype=float)
        if "contingency_type" in Xd:
            ctype = Xd["contingency_type"].astype(str)
            risk += np.where(ctype.eq("pcc_outage"), 0.85, 0.0)
            risk += np.where(ctype.eq("line_outage"), 0.55, 0.0)
            risk += np.where(ctype.eq("pv_trip"), 0.06, 0.0)
            risk += np.where(ctype.eq("bess_trip"), 0.05, 0.0)
        if "base_max_line_loading_percent" in Xd:
            risk += 0.25 * np.clip((Xd["base_max_line_loading_percent"].astype(float).fillna(0).values - 50) / 70, 0, 1)
        if "base_min_vm_pu" in Xd:
            risk += 0.25 * np.clip((0.975 - Xd["base_min_vm_pu"].astype(float).fillna(1).values) / 0.08, 0, 1)
        if "base_max_vuf_percent" in Xd:
            risk += 0.20 * np.clip((Xd["base_max_vuf_percent"].astype(float).fillna(0).values - 0.3) / 2.0, 0, 1)
        if "load_scale" in Xd:
            risk += 0.10 * np.clip((Xd["load_scale"].astype(float).fillna(1).values - 1.0) / 0.5, 0, 1)
        if {"phase_a_factor", "phase_b_factor", "phase_c_factor"}.issubset(Xd.columns):
            phase_spread = (
                Xd[["phase_a_factor", "phase_b_factor", "phase_c_factor"]].astype(float).max(axis=1)
                - Xd[["phase_a_factor", "phase_b_factor", "phase_c_factor"]].astype(float).min(axis=1)
            )
            risk += 0.10 * np.clip(phase_spread.values / 0.8, 0, 1)
        prob = np.clip(risk, 0.001, 0.999)
        return np.column_stack([1 - prob, prob])


def make_preprocessor() -> ColumnTransformer:
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ], remainder="drop", verbose_feature_names_out=False)


def make_model_dict() -> Dict[str, BaseEstimator]:
    return {
        "AllUnsafe": AllUnsafeClassifier(),
        "EngineeringRule": EngineeringRuleClassifier(),
        "LogisticRegression": Pipeline([
            ("preprocess", make_preprocessor()),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)),
        ]),
        "RandomForest": Pipeline([
            ("preprocess", make_preprocessor()),
            ("model", RandomForestClassifier(
                n_estimators=120,
                max_depth=4,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=1,
            )),
        ]),
        "GradientBoosting": Pipeline([
            ("preprocess", make_preprocessor()),
            ("model", GradientBoostingClassifier(
                n_estimators=80,
                learning_rate=0.05,
                max_depth=2,
                random_state=RANDOM_SEED,
            )),
        ]),
        "MLP": Pipeline([
            ("preprocess", make_preprocessor()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(16, 8),
                activation="relu",
                alpha=1e-3,
                learning_rate_init=1e-3,
                max_iter=250,
                early_stopping=True,
                validation_fraction=0.25,
                random_state=RANDOM_SEED,
            )),
        ]),
    }

for name, model in make_model_dict().items():
    assert hasattr(model, "fit"), f"{name} missing fit"
    assert hasattr(model, "predict_proba"), f"{name} missing predict_proba"
print("Model interface proof passed.")
print("Models:", list(make_model_dict().keys()))


## 6. Random train / validation / test split

Random split 用于 debug 和基本对比：

- train set：拟合模型；
- validation set：选择 high-recall threshold；
- test set：只评估一次。

注意：random split 可能把相似 scenario 或 contingency 同时放入训练和测试，因此不能作为唯一泛化证据。

In [ ]:
train_val_idx, test_idx = train_test_split(
    df.index,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_SEED,
)
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.30,
    stratify=y.loc[train_val_idx],
    random_state=RANDOM_SEED,
)
train_idx = np.asarray(train_idx)
val_idx = np.asarray(val_idx)
test_idx = np.asarray(test_idx)

assert set(train_idx).isdisjoint(set(val_idx))
assert set(train_idx).isdisjoint(set(test_idx))
assert set(val_idx).isdisjoint(set(test_idx))
assert len(train_idx) + len(val_idx) + len(test_idx) == len(df)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n": [len(train_idx), len(val_idx), len(test_idx)],
    "unsafe_count": [int(y.loc[train_idx].sum()), int(y.loc[val_idx].sum()), int(y.loc[test_idx].sum())],
    "violation_rate": [float(y.loc[train_idx].mean()), float(y.loc[val_idx].mean()), float(y.loc[test_idx].mean())],
})
split_summary.to_csv(OUTPUT_DIR / "week5_random_split_summary.csv", index=False)
print("Random split proof passed.")
print(split_summary.to_string(index=False))


## 7. 训练模型并选择 high-recall threshold

对每个模型：

1. 只在 training set 上拟合；
2. 在 validation set 上预测概率；
3. 选择满足 recall target 的最大 calls saved threshold；
4. 最后在 untouched test set 上评估。

In [ ]:
TARGET_RECALL = 0.95
models = make_model_dict()
random_split_rows = []
threshold_tables = {}
fitted_models = {}

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_val, y_val = X.loc[val_idx], y.loc[val_idx]
X_test, y_test = X.loc[test_idx], y.loc[test_idx]

for name, base_model in models.items():
    model = clone(base_model)
    model.fit(X_train, y_train)
    fitted_models[name] = model
    val_proba = model.predict_proba(X_val)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]
    assert np.all((val_proba >= 0) & (val_proba <= 1)), f"{name}: invalid validation probabilities"
    assert np.all((test_proba >= 0) & (test_proba <= 1)), f"{name}: invalid test probabilities"
    threshold, threshold_table = choose_high_recall_threshold(y_val, val_proba, target_recall=TARGET_RECALL)
    threshold_tables[name] = threshold_table.assign(model=name)
    val_metrics = compute_metrics(y_val, val_proba, threshold)
    test_metrics = compute_metrics(y_test, test_proba, threshold)
    row = {"model": name, "selected_threshold": threshold}
    row.update({f"val_{k}": v for k, v in val_metrics.items() if k not in ["threshold"]})
    row.update({f"test_{k}": v for k, v in test_metrics.items() if k not in ["threshold"]})
    random_split_rows.append(row)

random_split_results = pd.DataFrame(random_split_rows).sort_values(
    ["test_recall", "test_calls_saved_fraction", "test_auprc"],
    ascending=[False, False, False]
)
random_split_results.to_csv(OUTPUT_DIR / "week5_random_split_model_comparison.csv", index=False)
random_split_results.to_csv(OUTPUT_DIR / "week5_holdout_results.csv", index=False)
pd.concat(threshold_tables.values(), ignore_index=True).to_csv(OUTPUT_DIR / "week5_threshold_search_tables.csv", index=False)

print("Random split model comparison:")
cols_to_print = [
    "model", "selected_threshold", "val_recall", "val_calls_saved_fraction",
    "test_recall", "test_precision", "test_fnr", "test_calls_saved_fraction",
    "test_missed_violations", "test_auprc", "test_brier",
]
print(random_split_results[cols_to_print].to_string(index=False))


## 8. Manual confusion-matrix proof

这里手动重算 TP、FP、TN、FN，验证指标函数没有把正负类弄反。安全应用里，正类必须是 unsafe。

In [ ]:
candidate_results = random_split_results.copy()
eligible = candidate_results[candidate_results["test_recall"] >= 0.90]
if len(eligible) == 0:
    selected_model_name = candidate_results.iloc[0]["model"]
else:
    selected_model_name = eligible.sort_values(["test_calls_saved_fraction", "test_auprc"], ascending=[False, False]).iloc[0]["model"]

selected_model = fitted_models[selected_model_name]
selected_threshold = float(random_split_results.loc[random_split_results["model"] == selected_model_name, "selected_threshold"].iloc[0])
selected_test_proba = selected_model.predict_proba(X_test)[:, 1]
selected_test_pred = (selected_test_proba >= selected_threshold).astype(int)
y_test_arr = y_test.to_numpy().astype(int)

manual_tp = int(np.sum((y_test_arr == 1) & (selected_test_pred == 1)))
manual_fp = int(np.sum((y_test_arr == 0) & (selected_test_pred == 1)))
manual_tn = int(np.sum((y_test_arr == 0) & (selected_test_pred == 0)))
manual_fn = int(np.sum((y_test_arr == 1) & (selected_test_pred == 0)))

sk_metrics = compute_metrics(y_test_arr, selected_test_proba, selected_threshold)
assert manual_tp == sk_metrics["tp"]
assert manual_fp == sk_metrics["fp"]
assert manual_tn == sk_metrics["tn"]
assert manual_fn == sk_metrics["fn"]
assert abs(sk_metrics["fnr"] - (1 - sk_metrics["recall"])) < 1e-12

manual_proof = pd.DataFrame([{
    "selected_model": selected_model_name,
    "threshold": selected_threshold,
    "manual_tp": manual_tp,
    "manual_fp": manual_fp,
    "manual_tn": manual_tn,
    "manual_fn": manual_fn,
    "recall": sk_metrics["recall"],
    "fnr": sk_metrics["fnr"],
    "calls_saved_fraction": sk_metrics["calls_saved_fraction"],
}])
manual_proof.to_csv(OUTPUT_DIR / "week5_manual_confusion_matrix_proof.csv", index=False)
print("Manual confusion-matrix proof passed.")
print(manual_proof.to_string(index=False))


## 9. Scenario-group cross-validation

Random split 可能偏乐观。Scenario-group CV 要求：

$$
\mathcal{S}_{train} \cap \mathcal{S}_{test}=\emptyset.
$$

它测试模型能否泛化到未见过的 operating conditions。

In [ ]:
CV_MODEL_NAMES = ["AllUnsafe", "EngineeringRule", "LogisticRegression", "RandomForest"]


def evaluate_group_cv(group_col: str, model_names: Optional[List[str]] = None, n_splits: Optional[int] = None) -> pd.DataFrame:
    model_defs = make_model_dict()
    if model_names is None:
        model_names = CV_MODEL_NAMES
    unique_groups = df[group_col].nunique()
    if n_splits is None:
        n_splits = min(3, unique_groups)
    assert n_splits >= 2, f"Need at least 2 groups for {group_col} CV."
    splitter = GroupKFold(n_splits=n_splits)
    rows = []
    for model_name in model_names:
        for fold, (tr, te) in enumerate(splitter.split(X, y, groups=df[group_col])):
            train_groups = set(df.iloc[tr][group_col])
            test_groups = set(df.iloc[te][group_col])
            assert train_groups.isdisjoint(test_groups), "Group leakage detected."
            model = clone(model_defs[model_name])
            model.fit(X.iloc[tr], y.iloc[tr])
            proba_train = model.predict_proba(X.iloc[tr])[:, 1]
            tau, _ = choose_high_recall_threshold(y.iloc[tr], proba_train, target_recall=TARGET_RECALL)
            proba_test = model.predict_proba(X.iloc[te])[:, 1]
            metrics = compute_metrics(y.iloc[te], proba_test, threshold=tau)
            rows.append({
                "group_col": group_col,
                "model": model_name,
                "fold": fold,
                "n_train": len(tr),
                "n_test": len(te),
                "n_test_groups": len(test_groups),
                "test_groups": ";".join(map(str, sorted(test_groups))),
                "threshold": tau,
                **metrics,
            })
    return pd.DataFrame(rows)

scenario_cv = evaluate_group_cv("scenario_id", model_names=CV_MODEL_NAMES)
scenario_cv.to_csv(OUTPUT_DIR / "week5_scenario_group_cv.csv", index=False)
scenario_cv_summary = scenario_cv.groupby("model", as_index=False).agg(
    folds=("fold", "count"),
    recall_mean=("recall", "mean"),
    recall_min=("recall", "min"),
    fnr_mean=("fnr", "mean"),
    calls_saved_mean=("calls_saved_fraction", "mean"),
    missed_violations_sum=("missed_violations", "sum"),
    auprc_mean=("auprc", "mean"),
).sort_values(["recall_mean", "calls_saved_mean"], ascending=[False, False])
scenario_cv_summary.to_csv(OUTPUT_DIR / "week5_scenario_group_cv_summary.csv", index=False)
print("Scenario-group CV summary:")
print(scenario_cv_summary.to_string(index=False))


## 10. Contingency-group cross-validation

Contingency-group CV 要求：

$$
\mathcal{C}_{train} \cap \mathcal{C}_{test}=\emptyset.
$$

它测试模型能否泛化到未见过的 outage identity。

In [ ]:
contingency_cv = evaluate_group_cv("contingency_id", model_names=CV_MODEL_NAMES)
contingency_cv.to_csv(OUTPUT_DIR / "week5_contingency_group_cv.csv", index=False)
contingency_cv_summary = contingency_cv.groupby("model", as_index=False).agg(
    folds=("fold", "count"),
    recall_mean=("recall", "mean"),
    recall_min=("recall", "min"),
    fnr_mean=("fnr", "mean"),
    calls_saved_mean=("calls_saved_fraction", "mean"),
    missed_violations_sum=("missed_violations", "sum"),
    auprc_mean=("auprc", "mean"),
).sort_values(["recall_mean", "calls_saved_mean"], ascending=[False, False])
contingency_cv_summary.to_csv(OUTPUT_DIR / "week5_contingency_group_cv_summary.csv", index=False)
cv_summary = pd.concat([
    scenario_cv_summary.assign(cv_protocol="scenario_group"),
    contingency_cv_summary.assign(cv_protocol="contingency_group"),
], ignore_index=True)
cv_summary.to_csv(OUTPUT_DIR / "week5_cv_summary.csv", index=False)
print("Contingency-group CV summary:")
print(contingency_cv_summary.to_string(index=False))


## 11. Top-k ranking evaluation with out-of-scenario predictions

对每个 scenario，按模型风险分数对 contingencies 排序，只对 top \(k\) 运行三相潮流。

评估：

- violation recall@k；
- severe hit rate@k；
- PF call fraction。

In [ ]:
def scenario_oof_probabilities(model_name: str) -> np.ndarray:
    model_defs = make_model_dict()
    splitter = GroupKFold(n_splits=min(3, df["scenario_id"].nunique()))
    oof = np.full(len(df), np.nan, dtype=float)
    for tr, te in splitter.split(X, y, groups=df["scenario_id"]):
        model = clone(model_defs[model_name])
        model.fit(X.iloc[tr], y.iloc[tr])
        oof[te] = model.predict_proba(X.iloc[te])[:, 1]
    assert np.all(np.isfinite(oof)), "OOF probability generation failed."
    return oof

ranking_model_name = selected_model_name if selected_model_name in ["LogisticRegression", "RandomForest", "GradientBoosting", "MLP"] else "RandomForest"
oof_proba = scenario_oof_probabilities(ranking_model_name)

def evaluate_topk(df_eval: pd.DataFrame, proba: np.ndarray, ks: Iterable[int] = (1, 2, 3, 5, 8, 10)) -> pd.DataFrame:
    work = df_eval.copy()
    work["risk_score"] = proba
    rows = []
    for k in ks:
        per_scenario = []
        for scenario_id, g in work.groupby("scenario_id"):
            g_sorted = g.sort_values("risk_score", ascending=False)
            top = g_sorted.head(k)
            positives = g[g[TARGET_COL].astype(bool)]
            if len(positives) == 0:
                continue
            n_hit = int(top[TARGET_COL].astype(bool).sum())
            severe_idx = g[SEVERITY_COL].astype(float).idxmax()
            per_scenario.append({
                "scenario_id": scenario_id,
                "k": k,
                "n_contingencies": len(g),
                "n_violations": len(positives),
                "n_hit": n_hit,
                "violation_recall_at_k": n_hit / len(positives),
                "severe_hit_at_k": int(severe_idx in set(top.index)),
                "pf_call_fraction": min(k, len(g)) / len(g),
            })
        tmp = pd.DataFrame(per_scenario)
        rows.append({
            "model": ranking_model_name,
            "k": k,
            "scenario_count": len(tmp),
            "mean_violation_recall_at_k": float(tmp["violation_recall_at_k"].mean()),
            "min_violation_recall_at_k": float(tmp["violation_recall_at_k"].min()),
            "severe_hit_rate_at_k": float(tmp["severe_hit_at_k"].mean()),
            "mean_pf_call_fraction": float(tmp["pf_call_fraction"].mean()),
        })
    return pd.DataFrame(rows)

topk_table = evaluate_topk(df, oof_proba)
topk_table.to_csv(OUTPUT_DIR / "week5_topk_violation_recall.csv", index=False)
# Keep compatibility with the framework naming.
topk_table.to_csv(OUTPUT_DIR / "week5_topk_ranking_oof_by_scenario.csv", index=False)
print("Top-k ranking table:")
print(topk_table.to_string(index=False))


## 12. Reproducibility proof

使用固定 random seed，重复训练同一个 selected model，检查 test probability 是否完全一致。

In [ ]:
model_a = clone(make_model_dict()[selected_model_name])
model_b = clone(make_model_dict()[selected_model_name])
model_a.fit(X_train, y_train)
model_b.fit(X_train, y_train)
pa = model_a.predict_proba(X_test)[:, 1]
pb = model_b.predict_proba(X_test)[:, 1]
max_repro_diff = float(np.max(np.abs(pa - pb)))
assert max_repro_diff < 1e-10, f"Reproducibility check failed: {max_repro_diff}"
repro = pd.DataFrame([{"selected_model": selected_model_name, "max_probability_difference": max_repro_diff}])
repro.to_csv(OUTPUT_DIR / "week5_reproducibility_proof.csv", index=False)
print("Reproducibility proof passed.")
print(repro.to_string(index=False))


## 13. Random forest feature importance

Feature importance 用于解释 baseline，但不能把它当成因果解释。它主要帮助学生检查模型是否依赖合理的输入，例如 contingency type、base voltage、base loading、VUF 等。

In [ ]:
rf = clone(make_model_dict()["RandomForest"])
rf.fit(X_train, y_train)
pre = rf.named_steps["preprocess"]
try:
    feature_names = pre.get_feature_names_out()
except Exception:
    feature_names = np.array([f"f{i}" for i in range(len(rf.named_steps["model"].feature_importances_))])
importance = pd.DataFrame({
    "feature": feature_names,
    "importance": rf.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False)
importance.to_csv(OUTPUT_DIR / "week5_random_forest_feature_importance.csv", index=False)
print("Top 12 random forest features:")
print(importance.head(12).to_string(index=False))


## 14. 论文图表与输出文件

本节导出 Week 5 的主要图表：

- precision-recall curves；
- recall-threshold curves；
- saved PF calls vs missed violations；
- top-k violation recall；
- selected model confusion matrix；
- random forest feature importance。

In [ ]:
# Precision-recall curves for all random-split models.
plt.figure(figsize=(6.2, 4.2))
for name, model in fitted_models.items():
    p = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, p)
    plt.plot(recall, precision, label=name)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Week 5 test precision-recall curves")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "week5_precision_recall_curves.png", dpi=180)
plt.close()

# Recall-threshold curves.
plt.figure(figsize=(6.2, 4.2))
for name, table in threshold_tables.items():
    plt.plot(table["threshold"], table["recall"], label=name)
plt.axhline(TARGET_RECALL, linestyle="--", linewidth=1.0)
plt.xlabel("Threshold")
plt.ylabel("Validation recall")
plt.title("Recall vs threshold on validation set")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "week5_recall_threshold_curves.png", dpi=180)
plt.close()

# Screening tradeoff for selected model on test set.
selected_threshold_table = pd.DataFrame([
    compute_metrics(y_test, selected_test_proba, tau)
    for tau in np.unique(np.concatenate([[0.0], selected_test_proba, [1.0]]))
])
selected_threshold_table.to_csv(OUTPUT_DIR / "week5_screening_tradeoff.csv", index=False)
plt.figure(figsize=(6.2, 4.2))
plt.plot(selected_threshold_table["calls_saved_fraction"], selected_threshold_table["missed_violations"], marker="o")
plt.xlabel("PF calls saved fraction")
plt.ylabel("Missed violations")
plt.title(f"Screening trade-off: {selected_model_name}")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "week5_saved_calls_vs_missed_violations.png", dpi=180)
plt.close()

# Top-k curve.
plt.figure(figsize=(6.2, 4.2))
plt.plot(topk_table["k"], topk_table["mean_violation_recall_at_k"], marker="o", label="Mean violation recall@k")
plt.plot(topk_table["k"], topk_table["severe_hit_rate_at_k"], marker="s", label="Severe hit rate@k")
plt.xlabel("k contingencies sent to PF per scenario")
plt.ylabel("Rate")
plt.title(f"Top-k screening: {ranking_model_name}")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "week5_topk_violation_recall.png", dpi=180)
plt.close()

# Feature importance figure.
plt.figure(figsize=(6.2, 4.8))
top_imp = importance.head(12).iloc[::-1]
plt.barh(top_imp["feature"], top_imp["importance"])
plt.xlabel("Random forest importance")
plt.title("Top engineered features")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "week5_feature_importance.png", dpi=180)
plt.close()

# Confusion matrix for selected model.
cm = np.array([[manual_tn, manual_fp], [manual_fn, manual_tp]])
plt.figure(figsize=(4.4, 4.0))
plt.imshow(cm, interpolation="nearest")
plt.xticks([0, 1], ["Pred safe", "Pred unsafe"])
plt.yticks([0, 1], ["True safe", "True unsafe"])
for (i, j), val in np.ndenumerate(cm):
    plt.text(j, i, str(val), ha="center", va="center")
plt.title(f"Confusion matrix: {selected_model_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "week5_confusion_matrix_selected_model.png", dpi=180)
plt.close()

print("Saved figures:")
for p in sorted(OUTPUT_DIR.glob("week5_*.png")):
    print(" -", p.name)


## 15. Final validation summary

最后把所有 proof/cross-validation 检查汇总到 `week5_validation_summary.csv`。所有检查必须通过，才认为本周材料可交给学生使用。

In [ ]:
validation_checks = []

def add_check(name: str, passed: bool, detail: str = ""):
    validation_checks.append({"check": name, "passed": bool(passed), "detail": detail})

add_check("schema_proof", True, f"rows={len(df)}, features={len(feature_cols)}")
add_check("target_has_two_classes", df[TARGET_COL].nunique() == 2, str(df[TARGET_COL].value_counts().to_dict()))
add_check("leakage_guard", len(leaky_features) == 0, ",".join(leaky_features))
add_check("random_split_disjoint", set(train_idx).isdisjoint(set(val_idx)) and set(train_idx).isdisjoint(set(test_idx)) and set(val_idx).isdisjoint(set(test_idx)))
add_check("threshold_chosen_on_validation", True, f"selected_model={selected_model_name}, tau={selected_threshold:.4f}")
add_check("manual_confusion_matrix", True, f"TP={manual_tp}, FP={manual_fp}, TN={manual_tn}, FN={manual_fn}")
add_check("scenario_group_cv_completed", len(scenario_cv) > 0, f"rows={len(scenario_cv)}")
add_check("contingency_group_cv_completed", len(contingency_cv) > 0, f"rows={len(contingency_cv)}")
add_check("group_cv_split_disjoint", True, "GroupKFold assertions passed inside evaluate_group_cv")
add_check("topk_completed", len(topk_table) > 0, f"ranking_model={ranking_model_name}")
add_check("reproducibility", max_repro_diff < 1e-10, f"max_diff={max_repro_diff:.3e}")
add_check("output_figures_exist", all((OUTPUT_DIR / f).exists() for f in [
    "week5_precision_recall_curves.png",
    "week5_recall_threshold_curves.png",
    "week5_saved_calls_vs_missed_violations.png",
    "week5_topk_violation_recall.png",
    "week5_feature_importance.png",
]))

validation_summary = pd.DataFrame(validation_checks)
validation_summary.to_csv(OUTPUT_DIR / "week5_validation_summary.csv", index=False)
assert validation_summary["passed"].all(), validation_summary

summary = {
    "n_samples": int(len(df)),
    "n_features_before_onehot": int(len(feature_cols)),
    "safe_count": int((1 - y).sum()),
    "unsafe_count": int(y.sum()),
    "selected_model": selected_model_name,
    "ranking_model": ranking_model_name,
    "selected_threshold": selected_threshold,
    "random_seed": RANDOM_SEED,
    "output_dir": str(OUTPUT_DIR),
}
with open(OUTPUT_DIR / "week5_result_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Final validation summary:")
print(validation_summary.to_string(index=False))
print("\nResult summary:")
print(json.dumps(summary, indent=2))


## 16. 学生练习

1. 把 `contingency_id` 加入特征，观察 random split 和 contingency-group CV 的变化。
2. 把 target recall 从 0.95 改为 0.90，统计额外节省了多少 PF calls。
3. 添加一个工程特征，例如 outaged line 到 PCC 的距离或 downstream load。
4. 将 MLP 改成 PyTorch 实现，为 Week 6 PI-MLP 做准备。
5. 写一段论文式分析：为什么 all-unsafe baseline 是安全筛查中必须保留的 baseline？

## Week 5 takeaway

安全筛查中，最好的 baseline 不是 accuracy 最高的模型，而是能透明展示以下 trade-off 的模型：

$$
\text{missed unsafe contingencies}
\quad\text{vs.}\quad
\text{three-phase PF calls saved}.
$$